# Step 5 — Multi-Sample Integration + tSNE/UMAP/Clustering

## The problem this step solves

You have 39 cleaned per-sample objects. Now you need to build a single atlas — one embedding where all 204,883 cells can be visualized and compared. Two problems stand in the way.

### Problem 1 — Batch effects

Even when samples are processed identically, systematic technical differences between sequencing runs cause cells from different batches to cluster by technical origin rather than cell type. If uncorrected, a naive merge shows samples separating on the UMAP by sample identity — a T cell from batch 1 sits far from a T cell from batch 2, even though they are the same cell type.

The good news from Step 4: in the Yang et al. study, pseudobulk clustering confirmed that **tissue, diet, and exercise were the dominant sources of variation — not batch**. This is why the paper used `merge` (simple concatenation) rather than aggressive integration. If you had found batch effects in Step 4, you would choose Harmony or scVI here. Checking before integration is what lets you make this decision confidently.

> **ML analogy:** Batch effects are domain shift — the same signal measured under different technical conditions. Harmony is domain adaptation: it adjusts the learned representations (PCA embeddings) to align the domains without modifying the raw features.

### Problem 2 — tSNE does not scale

Standard tSNE scales as O(n²) in memory and time. At 204,883 cells it would take hours and gigabytes. FIt-SNE (Fourier-interpolated tSNE) solves this by approximating the repulsive force calculation using the Fast Fourier Transform, reducing complexity to O(n). It is the same algorithm, with the same interpretable output, just fast enough to run on hundreds of thousands of cells.

Beyond speed, the paper's tSNE implementation follows Kobak & Berens (2019, *Nature Commun.*, "The art of using t-SNE") which showed that initialization from PCA produces **more reproducible and globally coherent layouts** than random initialization. This is why the atlas tSNE in Figure 3A looks "clean" — the initialization, learning rate, and perplexity choices all matter.

## Why this matters for the Yang et al. paper

The integrated atlas (Figure 3A) is the foundation for every downstream result. The **22 annotated cell types** — including the three ASC states (IPC, CP, CD142+) and seven FAP states — were identified by examining which genes were differentially expressed within the clusters that emerged from this integration step. If the integration was poor (over-corrected batch effects, under-resolved clusters, wrong perplexity), the cell type assignments would be wrong and the paper's central findings about MSC responses to exercise would not hold.

The paper ran **4 integration objects**: one full atlas (all tissues) and one per tissue (scWAT, vWAT, SkM). Per-tissue objects resolve finer structure within a tissue because the embedding is not diluted by inter-tissue differences. The full atlas gives cross-tissue comparison.

## What this notebook does

1. Load all per-sample `.h5ad` files and concatenate on the intersection of genes
2. Apply post-merge filters: mitochondrial cutoff, optional proliferating cell removal
3. Normalize and select highly variable genes
4. Apply selected integration method: `merge` / Harmony / scVI
5. Run PCA → FIt-SNE (via openTSNE) → UMAP → Leiden + DBSCAN clustering
6. Save QC feature plots and the combined `.h5ad`

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from pathlib import Path

# openTSNE: Python implementation of FIt-SNE
# pip install openTSNE
from openTSNE import TSNE as openTSNE

# harmonypy: Python port of Harmony batch correction
# pip install harmonypy
import harmonypy as hm

## Configuration

**Integration method:**
- `merge` — simple concatenation; use only when samples come from the same batch or when you want to inspect uncorrected data
- `harmony` — Harmony batch correction on the PCA embedding; equivalent to Seurat's `rPCA`; the recommended default for most multi-sample experiments
- `scvi` — deep generative model (scVI); more powerful than Harmony for large, complex datasets but requires GPU or significant CPU time; equivalent to Seurat's `CCA` in intent

**`integrate_var`:** the `.obs` column that defines batches (e.g., `"sample_ID"`, `"library_ID"`). Harmony corrects for variance attributable to this variable.

**`rm_mt_hsp`:** removes mitochondrial and heat shock protein genes from the highly variable gene list before PCA. These genes are highly variable for technical reasons (stress during dissociation, cell death) rather than biological ones. Including them can cause cells to cluster by dissociation quality rather than cell type.

**`rm_prolif`:** removes actively dividing cells (Mki67+ or Pcna+). Proliferating cells from any lineage share a common transcriptional signature (cell cycle genes) that can cause them to cluster together regardless of their actual cell type identity, creating artifactual "proliferating" clusters that obscure the biology.

In [ ]:
input_file_list      = sorted(Path(".").glob("*_processed.h5ad"))
integration_method   = "harmony"   # "merge", "harmony", or "scvi"
integrate_var        = "sample_ID" # .obs column defining batches
df_name              = "all_tissues"
target_folder        = Path(".")

# Gene filtering
feature_cutoff = 0.001   # keep genes detected in at least 0.1% of cells
rm_mt_hsp      = False   # remove mt/heat-shock genes from variable features
species        = "mouse" # "mouse" or "human"

# Post-merge cell filtering
mt_cutoff  = 20.0  # upper bound on percent_mt after merge
rm_prolif  = True  # remove Mki67+ / Pcna+ proliferating cells

# Normalization
norm_mode = "regular"   # "regular" or "sctransform"

# Dimensionality reduction and clustering
dims = 50    # PCs used for UMAP, neighbors, clustering
res  = 0.4  # Leiden resolution; also eps for DBSCAN on UMAP

# Output directories
(target_folder / "plots").mkdir(exist_ok=True)
(target_folder / "rdata").mkdir(exist_ok=True)

mt_prefix   = "mt-" if species == "mouse" else "MT-"
prolif_gene = "Mki67" if species == "mouse" else "MKI67"
pcna_gene   = "Pcna"  if species == "mouse" else "PCNA"

## Step 1 — Load and concatenate samples

Each sample was processed independently in Step 3. The raw count matrices may not share exactly the same gene set if different CellRanger references or versions were used. We take the **intersection of genes** across all samples to ensure a consistent feature space.

This mirrors the R code's `cbind` with `intersect(rownames(...))` — only genes present in every sample are retained. Genes detected in very few cells are also dropped via `feature_cutoff` (default 0.1%), as they contribute noise without information.

In [ ]:
print(f"Loading {len(list(input_file_list))} samples...")
adatas = []
for fpath in input_file_list:
    adata = sc.read_h5ad(fpath)
    # Use raw counts for integration (stored in layers["counts"] by Step 3)
    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()
    print(f"  {fpath.stem}: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
    adatas.append(adata)

# Concatenate on the intersection of genes across all samples.
# join="inner" takes the gene intersection; label="sample" adds a
# column to .obs identifying each cell's source object.
combined = ad.concat(adatas, join="inner", label="source_file",
                     keys=[f.stem for f in input_file_list])
combined.var_names_make_unique()

print(f"\nCombined (pre-filter): {combined.n_obs:,} cells × {combined.n_vars:,} genes")

# Drop genes detected in fewer than feature_cutoff fraction of cells.
# These are so rare they cannot drive meaningful clustering but add
# sparsity and memory overhead.
min_cells = max(1, round(feature_cutoff * combined.n_obs))
sc.pp.filter_genes(combined, min_cells=min_cells)
print(f"After gene filter (min {min_cells} cells): {combined.n_vars:,} genes retained")

## Step 2 — Post-merge QC filtering

### Mitochondrial cutoff

A second mitochondrial filter is applied after merging. Some cells that passed the per-sample threshold in Step 3 may look worse in the context of the full dataset. Setting a consistent `mt_cutoff` across all samples also removes any cells that slipped through per-sample filters due to different thresholds.

### Proliferating cell removal

Mki67 (mouse) / MKI67 (human) is expressed exclusively during active cell division (G2/M phase). A cell that expresses Mki67 has a transcriptome dominated by the cell cycle program, which is shared across all proliferating cell types regardless of lineage.

The practical consequence: without removal, proliferating adipocyte precursors, proliferating immune cells, and proliferating muscle satellite cells will all cluster together in a "proliferating" supercluster, obscuring their true identities. PCNA is used as a fallback marker if Mki67 is not in the dataset.

In [ ]:
# Recompute percent_mt on the merged object
combined.var["mt"] = combined.var_names.str.startswith(mt_prefix)
sc.pp.calculate_qc_metrics(
    combined, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)

n_before = combined.n_obs

if rm_prolif:
    if prolif_gene in combined.var_names:
        marker = prolif_gene
    elif pcna_gene in combined.var_names:
        marker = pcna_gene
        print(f"  {prolif_gene} not found; using {pcna_gene} as proliferation marker")
    else:
        marker = None
        print("  No proliferation marker found; skipping proliferating cell removal")

    if marker:
        # Get expression of the proliferation marker per cell
        marker_idx = combined.var_names.get_loc(marker)
        if sp.issparse(combined.X):
            marker_expr = np.array(combined.X[:, marker_idx].todense()).flatten()
        else:
            marker_expr = combined.X[:, marker_idx]

        keep = (marker_expr == 0) & (combined.obs["pct_counts_mt"] < mt_cutoff)
        combined = combined[keep].copy()
        print(f"  Removed {n_before - combined.n_obs:,} proliferating cells "
              f"({marker}+) or high-mt cells")
else:
    keep = combined.obs["pct_counts_mt"] < mt_cutoff
    combined = combined[keep].copy()
    print(f"  Removed {n_before - combined.n_obs:,} high-mt cells")

print(f"Cells after post-merge filter: {combined.n_obs:,}")

## Step 3 — Normalization and highly variable gene selection

Normalization is applied to the merged object. When the integration method is `harmony` or `scvi`, normalization happens before integration (the batch correction is applied to normalized values, not raw counts). For `merge`, normalization is the only harmonization step.

### Removing mt and heat shock genes from variable features

If `rm_mt_hsp = True`, mitochondrial genes and heat shock protein genes are excluded from the highly variable gene list. These genes are highly variable across cells, but for reasons that are almost entirely technical:
- **Mitochondrial genes** vary with cell viability and dissociation stress, not cell identity
- **Heat shock protein genes** (Hsp90, Hspa1a, etc.) are induced by the stress of tissue dissociation itself — they are not differentially expressed between cell types in vivo

Including them in PCA would cause cells to cluster by dissociation quality or stress response rather than by their actual transcriptional identity. The heat shock gene list is species-specific and loaded from a reference file.

In [ ]:
# Store raw counts before normalization
combined.layers["counts"] = combined.X.copy()

if norm_mode == "regular":
    target_sum = float(np.median(
        np.array(combined.X.sum(axis=1)).flatten()
    ))
    sc.pp.normalize_total(combined, target_sum=target_sum)
    sc.pp.log1p(combined)
    sc.pp.scale(combined, max_value=10)
elif norm_mode == "sctransform":
    # Approximate SCTransform: normalize → log1p → scale
    # For a faithful port use rpy2 with the R sctransform package.
    sc.pp.normalize_total(combined)
    sc.pp.log1p(combined)
    sc.pp.scale(combined, max_value=10)

# Identify highly variable genes on the merged, normalized object
sc.pp.highly_variable_genes(combined, n_top_genes=2000, batch_key=integrate_var
                             if integrate_var in combined.obs.columns else None)

if rm_mt_hsp:
    # Exclude mitochondrial genes from the variable gene set
    mt_genes = combined.var_names[combined.var_names.str.startswith(mt_prefix)].tolist()

    # Load heat shock protein gene list if available
    hsp_list_path = Path(f"heat_shock_protein_gene_list_{species}.csv")
    if hsp_list_path.exists():
        hsp_genes = pd.read_csv(hsp_list_path, header=None)[0].tolist()
    else:
        # Fallback: common HSP gene prefixes
        hsp_prefixes = ("Hsp", "Hspa", "Hspb", "Hspc", "Hspd", "Hspe", "Hsph") \
                       if species == "mouse" else \
                       ("HSP", "HSPA", "HSPB", "HSPC", "HSPD", "HSPE", "HSPH")
        hsp_genes = combined.var_names[
            combined.var_names.str.startswith(hsp_prefixes)
        ].tolist()
        print(f"  HSP list file not found; excluded {len(hsp_genes)} genes "
              f"matching HSP prefixes")

    exclude = set(mt_genes) | set(hsp_genes)
    combined.var["highly_variable"] = (
        combined.var["highly_variable"] & ~combined.var_names.isin(exclude)
    )
    print(f"  Removed {len(exclude)} mt/HSP genes from variable features; "
          f"{combined.var['highly_variable'].sum()} variable genes remain")

print(f"Highly variable genes: {combined.var['highly_variable'].sum()}")

## Step 4 — Batch correction / integration

### Harmony (equivalent to Seurat rPCA)

Harmony (Korsunsky et al., 2019, *Nature Methods*) operates on the PCA embedding. After running PCA on all cells jointly, the PC coordinates reflect both biological variation and batch-driven variation. Harmony iteratively:

1. Clusters cells in PC space using a soft k-means
2. For each cluster, computes how much each batch deviates from the cluster centroid
3. Adds a correction vector to each cell's PC coordinates to remove the batch deviation
4. Repeats until convergence

The result is a corrected embedding (`X_pca_harmony`) where cells from the same biological state but different batches are pulled together. Crucially, the **gene expression matrix is not modified** — only the low-dimensional embedding used for UMAP and clustering is corrected. This means differential expression analysis can still be done on the original counts.

### Seurat rPCA vs. Harmony

Seurat's rPCA finds "anchor" cells — pairs of cells across samples that are mutual nearest neighbors in each sample's own PCA space — and uses them to estimate a batch transformation. Harmony is conceptually different (embedding correction vs. anchor-based alignment) but produces similar results in practice and is substantially faster.

In [ ]:
# PCA is run on highly variable genes regardless of integration method
sc.tl.pca(combined, n_comps=dims, use_highly_variable=True)
print(f"PCA complete: {dims} components")

if integration_method == "merge":
    # No batch correction — use the raw PCA embedding for UMAP and clustering
    print("Integration method: merge (no batch correction)")
    combined.obsm["X_pca_integrated"] = combined.obsm["X_pca"]

elif integration_method == "harmony":
    # Harmony: correct the PCA embedding for batch effects.
    # integrate_var must be a column in .obs that labels each cell's batch.
    if integrate_var not in combined.obs.columns:
        raise ValueError(
            f"integrate_var='{integrate_var}' not found in .obs columns: "
            f"{combined.obs.columns.tolist()}"
        )
    print(f"Running Harmony correction on '{integrate_var}'...")
    ho = hm.run_harmony(
        combined.obsm["X_pca"],
        combined.obs,
        integrate_var,
        max_iter_harmony=20,
        random_state=42,
    )
    combined.obsm["X_pca_harmony"]    = ho.Z_corr.T
    combined.obsm["X_pca_integrated"] = combined.obsm["X_pca_harmony"]
    print("Harmony correction complete")

elif integration_method == "scvi":
    # scVI: deep generative model for integration.
    # Requires: pip install scvi-tools
    # scVI learns a latent space that is batch-free by conditioning
    # the VAE encoder/decoder on the batch label. It is more powerful
    # than Harmony for large, complex datasets but is slower and requires
    # more memory. It also modifies the latent space (not PCA), so the
    # integrated embedding is not directly interpretable as PCs.
    try:
        import scvi
    except ImportError:
        raise ImportError("scvi-tools not installed. Run: pip install scvi-tools")

    print("Running scVI integration...")
    combined.layers["counts_int"] = combined.layers["counts"].copy()
    scvi.model.SCVI.setup_anndata(combined, layer="counts_int", batch_key=integrate_var)
    model = scvi.model.SCVI(combined, n_latent=30)
    model.train(max_epochs=400, early_stopping=True)
    combined.obsm["X_scvi"]           = model.get_latent_representation()
    combined.obsm["X_pca_integrated"] = combined.obsm["X_scvi"]
    print("scVI integration complete")

## Step 5 — FIt-SNE via openTSNE (fancy_tsne)

This section implements the `fancy_tsne.R` helper in Python using `openTSNE`.

### Why FIt-SNE instead of standard tSNE?

Standard tSNE computes repulsive forces between all pairs of cells, scaling as O(n²). Barnes-Hut tSNE approximates this as O(n log n) but is still slow above 50,000 cells and takes hours for 200,000. FIt-SNE (Linderman et al., 2019, *Nature Methods*) uses the Fast Fourier Transform to approximate the repulsive forces in O(n) time, making 200,000+ cells practical.

### PCA initialization

Standard tSNE starts from random noise. Kobak & Berens (2019) showed that initializing from the first two PCs produces:
- **Better global structure** — the large-scale arrangement of clusters in the tSNE reflects the PCA topology, making it easier to interpret
- **Reproducibility** — the layout is deterministic rather than sensitive to the random seed
- **Faster convergence** — fewer iterations are needed because the initialization is already close to the final solution

The R code scales the PCA initialization by `sd(PC1) * 0.0001` to make the initial spread very small, then lets tSNE expand it naturally.

### Multi-scale perplexity

Perplexity controls the effective number of neighbors each cell considers when computing similarities. A single value creates a trade-off: low perplexity (e.g., 5) captures fine local structure but loses global coherence; high perplexity captures global structure but blurs local clusters. Using two perplexities simultaneously (`perplexity_list = [30, n/100]`) preserves both scales. This is used only when `n < 100,000`; for very large datasets a single perplexity of 30 is used to keep runtime tractable.

In [ ]:
n_cells      = combined.n_obs
very_large   = n_cells > 100_000
learning_rate = n_cells / 12

# PCA-based initialization scaled to a small variance, matching the R code:
# PCA.init <- pca[, 1:2] / sd(pca[, 1]) * 0.0001
pca_embed = combined.obsm["X_pca_integrated"]
pca_init  = pca_embed[:, :2] / pca_embed[:, 0].std() * 0.0001

print(f"Running openTSNE on {n_cells:,} cells "
      f"({'very large — single perplexity' if very_large else 'multi-scale perplexity'})...")

if very_large:
    # Single perplexity with exaggeration — matches R:
    # fftRtsne(..., perplexity=30, exaggeration_factor=4)
    tsne = openTSNE(
        perplexity=30,
        initialization=pca_init,
        learning_rate=learning_rate,
        n_iter=1000,
        exaggeration=4,
        negative_gradient_method="fft",
        random_state=42,
        n_jobs=-1,
        verbose=True,
    )
else:
    # Multi-scale perplexity — matches R:
    # fftRtsne(..., perplexity_list=c(30, floor(n/100)), perplexity=0)
    perp2 = max(10, int(n_cells / 100))
    tsne = openTSNE(
        perplexity=[30, perp2],
        initialization=pca_init,
        learning_rate=learning_rate,
        n_iter=1000,
        negative_gradient_method="fft",
        random_state=42,
        n_jobs=-1,
        verbose=True,
    )

tsne_embedding = tsne.fit(pca_embed)
combined.obsm["X_tsne"] = np.array(tsne_embedding)
print("tSNE complete")

## Step 6 — UMAP, neighbor graph, and clustering

### UMAP

UMAP (McInnes et al., 2018) is computed in addition to tSNE. The two embeddings complement each other:
- tSNE (with PCA init and multi-scale perplexity) better preserves **local** neighborhood structure and tends to produce cleaner cluster separation
- UMAP better preserves **global** distances and is faster to compute on very large datasets

Both are kept so downstream analysis (Step 7) can use whichever is more informative for a given visualization.

### Leiden clustering

Leiden (Traag et al., 2019) improves on Louvain by guaranteeing that all clusters are internally connected. It operates on the k-nearest-neighbor graph built in PCA space. The `resolution` parameter controls granularity — the value used here is provisional; final cluster resolution is tuned in Step 7 after cell type annotation.

### DBSCAN on UMAP

DBSCAN (density-based clustering) is run on the 2D UMAP coordinates as a complementary clustering approach. Unlike Leiden, DBSCAN does not require specifying the number of clusters — it finds clusters as regions of high point density and labels low-density regions as noise (cluster 0). This is useful for identifying isolated rare populations that may be hard to resolve with graph-based methods, and for flagging cells that do not belong to any coherent cluster.

In [ ]:
# Build neighbor graph on the integrated PCA embedding
sc.pp.neighbors(combined, use_rep="X_pca_integrated", n_pcs=dims)

# UMAP
sc.tl.umap(combined)
print("UMAP complete")

# Leiden clustering
sc.tl.leiden(combined, resolution=res, key_added="leiden")
print(f"Leiden clusters: {combined.obs['leiden'].nunique()} "
      f"(resolution={res})")

# DBSCAN on UMAP coordinates
# eps matches the Leiden resolution used in R: dbscan(umap_embed, eps=res)
umap_coords = combined.obsm["X_umap"]
dbscan_labels = DBSCAN(eps=res, min_samples=5).fit_predict(umap_coords)
combined.obs["umap_dbscan"] = dbscan_labels.astype(str)
n_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise  = (dbscan_labels == -1).sum()
print(f"DBSCAN clusters: {n_dbscan} ({n_noise:,} noise cells labeled -1)")

## Step 7 — QC feature plots

After integration, a quick visual check confirms that the combined object looks biologically reasonable before saving. The R script plots five features on the UMAP:

- **Ptprc (CD45)** — pan-immune marker. Should be expressed in immune cell clusters and absent from adipocytes, muscle fibers, and stromal cells.
- **Hba-a1** — hemoglobin alpha chain. Should be expressed only in red blood cells (or erythroid precursors). If broadly expressed, SoupX contamination may have been insufficient.
- **nCount_RNA** — total UMI count. Should be relatively uniform across the UMAP; strong gradients may indicate residual depth effects not corrected by normalization.
- **nFeature_RNA** — gene count. Similar to nCount — uniform is good.
- **percent_mt** — mitochondrial fraction. Should be low and uniform; a cluster with high percent_mt may be damaged cells that slipped through filtering.

In [ ]:
qc_genes = [
    "Ptprc" if species == "mouse" else "PTPRC",
    "Hba-a1" if species == "mouse" else "HBA1",
]
qc_obs   = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]

# Filter to features actually present in the dataset
qc_genes_present = [g for g in qc_genes if g in combined.var_names]
all_features     = qc_genes_present + qc_obs

fig, axes = plt.subplots(1, len(all_features), figsize=(4 * len(all_features), 4))
umap_xy = combined.obsm["X_umap"]

for ax, feat in zip(axes, all_features):
    if feat in combined.var_names:
        idx = combined.var_names.get_loc(feat)
        vals = (combined.X[:, idx].toarray().flatten()
                if sp.issparse(combined.X) else combined.X[:, idx])
    else:
        vals = combined.obs[feat].values

    sc_plot = ax.scatter(umap_xy[:, 0], umap_xy[:, 1],
                         c=vals, s=0.5, cmap="viridis", rasterized=True)
    ax.set_title(feat, fontsize=9)
    ax.axis("off")
    plt.colorbar(sc_plot, ax=ax, shrink=0.6)

plt.suptitle(f"{df_name} — QC check", fontsize=11)
plt.tight_layout()
qc_path = target_folder / "plots" / f"{df_name}_QC_check.png"
fig.savefig(qc_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {qc_path}")

## Step 8 — Save combined object

The combined, integrated object is saved as `.h5ad`. This file is the input for Step 6 (`add_metadata.R`) and Step 7 (`analysis_pipeline_scwat_vwat_skm.Rmd`).

The `.h5ad` replaces the `.qs` format used in the R pipeline — both are efficient binary formats for serializing large annotated matrices. Unlike `.qs`, `.h5ad` is readable by both Python (via `anndata`) and R (via `anndata` or `SeuratDisk`), making it portable across tools.

In [ ]:
out_path = target_folder / "rdata" / f"{df_name}_combined.h5ad"
combined.write_h5ad(out_path)
print(f"Saved: {out_path}")
print(f"Final object: {combined.n_obs:,} cells × {combined.n_vars:,} genes")
print(f"Embeddings:   {list(combined.obsm.keys())}")
print(f"Obs columns:  {combined.obs.columns.tolist()}")